In [11]:
import json
import os
from pathlib import Path
# from DataSplit import DataSplit

with open ('Datasets/R_T.json', 'r') as f:
    ogData = json.load(f)

with open ('Datasets/Waleed.coco/train/_annotations.coco.json', 'r') as f2:
    incomingData = json.load(f2)

with open('Garbage_detection.coco/train/_annotations.coco.json') as f3:
    data = json.load(f3)
# incomingData

In [12]:
# !New_problems!
#What if we add an image that already exists?
#I think we should look at the image>extra>name || images>file_name
#Going for a loop, and if same name found, *we have to store its id* and delete that image from the incoming data.
##Then minus each incoming images["id"] by one.

# Which causes another problem in mapping.
#1. in annotations we have to delete any annotation that has the same annotations{"image_id"} as the deleted one.
#2. After that we have to minus the number of annotations of that deleted image.

In [13]:
def del_background_category(data):
    categories = data['categories']
    new_categories = []
    for category in categories:
        if category['id'] == 0:
            continue
        new_categories.append(category)

    data['categories'] = new_categories
    updated_json = json.dumps(data, indent=4)
    return updated_json

# deleted = del_background_category(data)
# with open("Garbage_detection.coco/train/_annotations.coco.json", "w") as f:
#         json.dump(data, f, indent=4)

In [14]:
#ogImgs
#incomingImgs
def check_duplicates(ogData, incomingData, train=True):

    ogImgs = ogData['images']
    incomingImgs = incomingData['images']

    OG_IMAGES = {}
    INCOMING_IMGs = {}
    duplicated_imgs = {}
    dub_found = False

    for image in ogImgs:
        # OG_IMAGES.append(image['file_name'])
        OG_IMAGES[image['extra'].get('name')] = image['id']

    for image in incomingImgs:
        # INCOMING_IMGs.append(image['file_name'])
        INCOMING_IMGs[image['extra'].get('name')] = image['id']

    for key, value in INCOMING_IMGs.items():
        if key in OG_IMAGES:
            # save_id.append(value)
            duplicated_imgs[key] = value
            dub_found = True

    # print(OG_IMAGES)
    # print(INCOMING_IMGs)
    # print(duplicated_imgs)

    og_file_name = []
    if dub_found:
        for key, value in duplicated_imgs.items():
            for image in incomingImgs:
                if key == image['extra'].get('name'):
                    og_file_name.append(image['file_name'])
                
    # print(og_file_name)
    if train:
        for i in range (len(og_file_name)):
            if train: os.remove(f'{og_file_name[i]}')

# check_duplicates(ogData, incomingData)

In [15]:

# incomingAnnotations = incomingData['annotations']

#The last element id in the base json file...
element_id_offset = len(ogData['annotations'])
new_annotate_img_offset = ogData['annotations'][-1].get('image_id') + 1

# new_element_id = []
# new_annotate_img_id = []

for ann in incomingData['annotations']:
    ann['id']+=element_id_offset
    ann['image_id']+=new_annotate_img_offset

    # new_element_id.append((ann['id'] + element_id_offset))
    # new_annotate_img_id.append((ann['image_id'] + new_annotate_img_offset))

In [16]:
ogImgs = ogData['images']
incomingImgs = incomingData['images']
idImageOffset = len(ogImgs)

# new_img_id = []
for img in incomingData['images']:
    
    img['id']+=idImageOffset
    # new_img_id.append((img['id']+ idImageOffset))

In [17]:

print(ogData['annotations'][-1].get('id'))
print(incomingData['annotations'][0].get('id'))

2696
2697


In [18]:
file_path = Path("R_T_W.json")

if not file_path.is_file():
    
    ogData["images"].extend(incomingData["images"])
    ogData["annotations"].extend(incomingData["annotations"])
    # new_categories = del_background_category(ogData)

    with open("Datasets/R_T_W.json", "w") as f:
        json.dump(ogData, f, indent=4)